In [1]:
import os

os.environ["PYSPARK_PYTHON"] = r"C:\Users\mahmo\AppData\Local\Programs\Python\Python311\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\mahmo\AppData\Local\Programs\Python\Python311\python.exe"

os.environ["PYSPARK_PYTHON_DRIVER"] = r"C:\Users\mahmo\AppData\Local\Programs\Python\Python311\python.exe"

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .getOrCreate()

print("Spark started 🚀")

Spark started 🚀


In [3]:
data = [
    (1, "Ahmed", 25, "M", "Diabetes", "Metformin", "2024-01-01"),
    (1, "Ahmed", 25, "M", "Hypertension", "Amlodipine", "2024-02-01"),
    (2, "Sara", 30, "F", "Asthma", "Ventolin", "2024-01-15"),
    (3, "Ali", 40, "M", "Diabetes", "Insulin", "2024-03-10"),
    (3, "Ali", 40, "M", "Diabetes", "Metformin", "2024-04-01")
]

columns = ["patient_id","name","age","gender","disease","drug","visit_date"]

df = spark.createDataFrame(data, columns)
df.show()

+----------+-----+---+------+------------+----------+----------+
|patient_id| name|age|gender|     disease|      drug|visit_date|
+----------+-----+---+------+------------+----------+----------+
|         1|Ahmed| 25|     M|    Diabetes| Metformin|2024-01-01|
|         1|Ahmed| 25|     M|Hypertension|Amlodipine|2024-02-01|
|         2| Sara| 30|     F|      Asthma|  Ventolin|2024-01-15|
|         3|  Ali| 40|     M|    Diabetes|   Insulin|2024-03-10|
|         3|  Ali| 40|     M|    Diabetes| Metformin|2024-04-01|
+----------+-----+---+------+------------+----------+----------+



In [4]:
df

DataFrame[patient_id: bigint, name: string, age: bigint, gender: string, disease: string, drug: string, visit_date: string]

In [5]:
df.printSchema()
df.show()


root
 |-- patient_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- disease: string (nullable = true)
 |-- drug: string (nullable = true)
 |-- visit_date: string (nullable = true)

+----------+-----+---+------+------------+----------+----------+
|patient_id| name|age|gender|     disease|      drug|visit_date|
+----------+-----+---+------+------------+----------+----------+
|         1|Ahmed| 25|     M|    Diabetes| Metformin|2024-01-01|
|         1|Ahmed| 25|     M|Hypertension|Amlodipine|2024-02-01|
|         2| Sara| 30|     F|      Asthma|  Ventolin|2024-01-15|
|         3|  Ali| 40|     M|    Diabetes|   Insulin|2024-03-10|
|         3|  Ali| 40|     M|    Diabetes| Metformin|2024-04-01|
+----------+-----+---+------+------------+----------+----------+



In [6]:
from pyspark.sql.functions  import to_date

In [7]:
df= df.withColumn("visit_date" , to_date("visit_date"))

In [8]:
df.printSchema()

root
 |-- patient_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- disease: string (nullable = true)
 |-- drug: string (nullable = true)
 |-- visit_date: date (nullable = true)



In [9]:
from pyspark.sql.functions import col , sum as _sum

In [10]:
df.select([
    _sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns]).show()

+----------+----+---+------+-------+----+----------+
|patient_id|name|age|gender|disease|drug|visit_date|
+----------+----+---+------+-------+----+----------+
|         0|   0|  0|     0|      0|   0|         0|
+----------+----+---+------+-------+----+----------+



In [11]:
df.groupBy(df.columns).count().filter("count > 1").show()

+----------+----+---+------+-------+----+----------+-----+
|patient_id|name|age|gender|disease|drug|visit_date|count|
+----------+----+---+------+-------+----+----------+-----+
+----------+----+---+------+-------+----+----------+-----+



In [12]:
# no of visits for every patients 
df.groupby("patient_id").count().show()


+----------+-----+
|patient_id|count|
+----------+-----+
|         1|    2|
|         3|    2|
|         2|    1|
+----------+-----+



In [13]:
# most frequent diseases
df.groupby("disease").count().orderBy("count", ascending=False).show()

+------------+-----+
|     disease|count|
+------------+-----+
|    Diabetes|    3|
|Hypertension|    1|
|      Asthma|    1|
+------------+-----+



In [14]:
# most drugs used
df.groupby("drug").count().orderBy("count",ascending=False).show()

+----------+-----+
|      drug|count|
+----------+-----+
| Metformin|    2|
|  Ventolin|    1|
|   Insulin|    1|
|Amlodipine|    1|
+----------+-----+



In [15]:
# average age per every disease
df.groupby("disease").avg("age").show()

+------------+--------+
|     disease|avg(age)|
+------------+--------+
|    Diabetes|    35.0|
|Hypertension|    25.0|
|      Asthma|    30.0|
+------------+--------+



In [16]:
from pyspark.sql.functions import countDistinct
df.groupby("disease").agg(countDistinct("drug").alias("drug_count")).show()

+------------+----------+
|     disease|drug_count|
+------------+----------+
|    Diabetes|         2|
|Hypertension|         1|
|      Asthma|         1|
+------------+----------+



In [17]:
# no of diseases for every patient 
from pyspark.sql.functions import countDistinct
df_disease= df.groupby("patient_id").agg(countDistinct("disease"))
df_visit= df.groupby("patient_id").count()
risk_df= df_disease.join(df_visit,"patient_id")
risk_df.show()

+----------+-----------------------+-----+
|patient_id|count(DISTINCT disease)|count|
+----------+-----------------------+-----+
|         1|                      2|    2|
|         3|                      1|    2|
|         2|                      1|    1|
+----------+-----------------------+-----+



In [18]:
# no of diseases for every patient 
from pyspark.sql.functions import countDistinct
df_disease= df.groupby("patient_id").agg(countDistinct("disease").alias("disease_count"))
df_visit= df.groupby("patient_id").count() .withColumnRenamed("count", "visit_count")
# every pt has how many diseases and how many visits 
risk_df= df_disease.join(df_visit,"patient_id")
risk_df.show()

+----------+-------------+-----------+
|patient_id|disease_count|visit_count|
+----------+-------------+-----------+
|         1|            2|          2|
|         3|            1|          2|
|         2|            1|          1|
+----------+-------------+-----------+



In [19]:
from pyspark.sql.functions import year , monthname

df= df.withColumn("year",year("visit_date"))
df= df.withColumn("monthname",monthname("visit_date"))
df.show()

+----------+-----+---+------+------------+----------+----------+----+---------+
|patient_id| name|age|gender|     disease|      drug|visit_date|year|monthname|
+----------+-----+---+------+------------+----------+----------+----+---------+
|         1|Ahmed| 25|     M|    Diabetes| Metformin|2024-01-01|2024|      Jan|
|         1|Ahmed| 25|     M|Hypertension|Amlodipine|2024-02-01|2024|      Feb|
|         2| Sara| 30|     F|      Asthma|  Ventolin|2024-01-15|2024|      Jan|
|         3|  Ali| 40|     M|    Diabetes|   Insulin|2024-03-10|2024|      Mar|
|         3|  Ali| 40|     M|    Diabetes| Metformin|2024-04-01|2024|      Apr|
+----------+-----+---+------+------------+----------+----------+----+---------+



In [20]:
from pyspark.sql.functions import countDistinct
df_disease_year= df.groupby("disease","year").agg({"disease":"count"})

df_disease_year.show()                                                            



+------------+----+--------------+
|     disease|year|count(disease)|
+------------+----+--------------+
|Hypertension|2024|             1|
|      Asthma|2024|             1|
|    Diabetes|2024|             3|
+------------+----+--------------+



In [21]:
# give the same result as above code
from pyspark.sql.functions import count
df_disease_Year= df.groupby("disease","year").agg(count("*").alias("disease_count"))
df_disease_Year.show()

+------------+----+-------------+
|     disease|year|disease_count|
+------------+----+-------------+
|Hypertension|2024|            1|
|      Asthma|2024|            1|
|    Diabetes|2024|            3|
+------------+----+-------------+



In [22]:
# no of diff drugs for every patient
from pyspark.sql.functions import countDistinct
df_paient_drug= df.groupby("patient_id").agg(countDistinct("drug").alias("drug_count"))
df_paient_drug.show()

+----------+----------+
|patient_id|drug_count|
+----------+----------+
|         1|         2|
|         3|         2|
|         2|         1|
+----------+----------+



In [23]:
patient_dim = df.select("patient_id","age","name","gender").dropDuplicates(["patient_id"])
patient_dim.show()

+----------+---+-----+------+
|patient_id|age| name|gender|
+----------+---+-----+------+
|         1| 25|Ahmed|     M|
|         2| 30| Sara|     F|
|         3| 40|  Ali|     M|
+----------+---+-----+------+



In [24]:
# ده زي الكود الفوق لكن ده بيجيب مريض واحد  بس احدث زياره ليه يعني
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("patient_id").orderBy(desc("visit_date"))

patient_dim = (
    df.withColumn("rn", row_number().over(window))
      .filter("rn = 1")
      .select("patient_id", "name", "age", "gender","rn")
)
patient_dim.show()

+----------+-----+---+------+---+
|patient_id| name|age|gender| rn|
+----------+-----+---+------+---+
|         1|Ahmed| 25|     M|  1|
|         2| Sara| 30|     F|  1|
|         3|  Ali| 40|     M|  1|
+----------+-----+---+------+---+



In [25]:
visits_fact = (
    df.select(
        "patient_id",
        "disease",
        "drug",
        "visit_date",
        "year",
        "monthname"
    )
)

visits_fact.show()

+----------+------------+----------+----------+----+---------+
|patient_id|     disease|      drug|visit_date|year|monthname|
+----------+------------+----------+----------+----+---------+
|         1|    Diabetes| Metformin|2024-01-01|2024|      Jan|
|         1|Hypertension|Amlodipine|2024-02-01|2024|      Feb|
|         2|      Asthma|  Ventolin|2024-01-15|2024|      Jan|
|         3|    Diabetes|   Insulin|2024-03-10|2024|      Mar|
|         3|    Diabetes| Metformin|2024-04-01|2024|      Apr|
+----------+------------+----------+----------+----+---------+



In [26]:
from pyspark.sql.functions import countDistinct
df_visits= df.groupby("patient_id").count().withColumnRenamed("count", "visit_count")
df_diseases= df.groupby("patient_id").agg(countDistinct("disease").alias("disease_count"))
df_drugs= df.groupby("patient_id").agg(countDistinct("drug").alias("drug_count"))
df_risk= df_visits.join(df_diseases,"patient_id")
df_risk= df_risk.join(df_drugs,"patient_id")
df_risk.show()

+----------+-----------+-------------+----------+
|patient_id|visit_count|disease_count|drug_count|
+----------+-----------+-------------+----------+
|         1|          2|            2|         2|
|         3|          2|            1|         2|
|         2|          1|            1|         1|
+----------+-----------+-------------+----------+



In [27]:
from pyspark.sql.functions import col, when

df_risk = df_risk.withColumn("risk_score",col("visit_count") + col("disease_count") + col("drug_count"))
    
df_risk=df_risk.withColumn("risk_level",when(col("risk_score") >= 6, "High")
        .when(col("risk_score") >= 3, "Medium")
        .otherwise("Low")
    )
df_risk.show()

+----------+-----------+-------------+----------+----------+----------+
|patient_id|visit_count|disease_count|drug_count|risk_score|risk_level|
+----------+-----------+-------------+----------+----------+----------+
|         1|          2|            2|         2|         6|      High|
|         3|          2|            1|         2|         5|    Medium|
|         2|          1|            1|         1|         3|    Medium|
+----------+-----------+-------------+----------+----------+----------+



In [28]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.orderBy("drug")

dim_drug = (
    df.select("drug")
      .dropDuplicates()
      .withColumn("drug_id", row_number().over(window))
)
dim_drug.show()


+----------+-------+
|      drug|drug_id|
+----------+-------+
|Amlodipine|      1|
|   Insulin|      2|
| Metformin|      3|
|  Ventolin|      4|
+----------+-------+



In [29]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window=Window.orderBy("disease")

dim_disease = df.select("disease").dropDuplicates().withColumn("disease_id",row_number().over(window))

dim_disease.show()



+------------+----------+
|     disease|disease_id|
+------------+----------+
|      Asthma|         1|
|    Diabetes|         2|
|Hypertension|         3|
+------------+----------+



In [30]:
visits_fact= df.join(dim_drug,"drug","left")
visits_fact= visits_fact.join(dim_disease,"disease","left")
visits_fact= visits_fact.select(
        "patient_id",
        "drug_id",
        "disease_id",
        "visit_date",
        "year",
        "monthname"
) 

visits_fact.show()

+----------+-------+----------+----------+----+---------+
|patient_id|drug_id|disease_id|visit_date|year|monthname|
+----------+-------+----------+----------+----+---------+
|         1|      3|         2|2024-01-01|2024|      Jan|
|         3|      3|         2|2024-04-01|2024|      Apr|
|         2|      4|         1|2024-01-15|2024|      Jan|
|         3|      2|         2|2024-03-10|2024|      Mar|
|         1|      1|         3|2024-02-01|2024|      Feb|
+----------+-------+----------+----------+----+---------+



In [31]:
type(visits_fact)

pyspark.sql.classic.dataframe.DataFrame

In [32]:
visits_fact.filter("drug_id is null OR disease_id is null").show()

+----------+-------+----------+----------+----+---------+
|patient_id|drug_id|disease_id|visit_date|year|monthname|
+----------+-------+----------+----------+----+---------+
+----------+-------+----------+----------+----+---------+



In [35]:
import sys
!{sys.executable} -m pip install pandas

     ---------------------------------------- 9.9/9.9 MB 8.4 MB/s eta 0:00:00
     --------------------------------------- 12.6/12.6 MB 15.2 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.1
[notice] To update, run: C:\Users\mahmo\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [36]:
import pandas as pd
print(pd.__version__)

3.0.2


In [38]:
visits_fact.toPandas().to_csv("visits_fact.csv", index=False)

In [39]:
patient_dim.toPandas().to_csv("patient_dim.csv", index=False)

In [40]:
dim_drug.toPandas().to_csv("dim_drug.csv", index=False)

In [41]:
dim_disease.toPandas().to_csv("dim_disease.csv", index=False)

In [43]:
df_risk.toPandas().to_csv("df_risk.csv", index=False)